In [ ]:
import pandas as pd, numpy as np, textstat, json, time 
import plotly_express as px

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer, util

In [ ]:
from evaluate import load

In [1]:
from sentence_transformers import SentenceTransformer, util

: 

In [5]:
xaiqb_human = {
    "questions": {
        "Data": {
                "What kind of data was the system trained on?": 
                {
                    "answer": "The data is tabular and provided in two csv files. One file contains the features including protected (e.g. sex, age, education, employment), behavioural (e.g. social distancing, mask wearing, hand washing), opinion (H1N1 concern, doctor vaccine reccomendation), health (e.g. chronic medial conditions, health insurance) and demographic (e.g. geographical region, employment industry) characteristics. The other file contains their labels/ground-truth which indicate if a respondant recieved the H1N1 vaccination or not (Yes = 1, No = 0)."
                },
                "What is the source of the training data?": 
                {
                    "answer": "Data is sourced from DrivenData's 'Flu Shot Learning: Predict H1N1 and Seasonal Flu Vaccines' competition and originates from the US National 2009 H1N1 Flu Survey (NHFS). The US National 2009 H1N1 Flu Survey (NHFS) was conducted between October 2009 and June 2010. It was a one-time telephone survey of households to monitor vaccinations during the 2009-2010 flu season in response to the 2009 H1N1 pandemic."
                },
                "How were the labels/ground-truth produced?": 
                {
                    "answer": "As part of the US National 2009 H1N1 Flu Survey (NHFS) a telephone sruvey was conducted. Respondants self reported if they had recieved the H1N1 vaccination. This is a binary feature indicating if they recieved the H1N1 vaccination or not (Yes = 1, No = 0)."
                },
                "What is the sample size of the training data?": 
                {
                    "answer": "There are a total of 26,707 observations with 37 features. This was split into training and testing data using an 80:20 split. Training data is 21,365 rows, testing data is 5,342 rows."
                },
                "What dataset(s) is the system NOT using?": 
                {
                    "answer": "The system is not using any additional data such as public health datasets or electronic health records. Only the survey data provided by the US National 2009 H1N1 Flu Survey (NHFS) is being used by the system."
                },
                "What are the potential limitations/biases of the data?": 
                {
                    "answer": "The interviews were conducted over telephone and all respondants were persons 6 months or older living in the United States. These respondants were self reporting on their vaccination status of the H1N1 vaccine. This limits the representation of the overall (global) population and the attitudes towards the H1N1 vaccination. This means the data has limited transferability, and cannot be used to effectively model other countries and/or demographics."
                }
        }
    }
}

xaiqb_chatgpt = {
  "questions": {
    "Data": {
        "What kind of data was the system trained on?": {
          "answer": "The system was trained on survey data from the DrivenData 'Flu Shot Learning: Predict H1N1 and Seasonal Flu Vaccines' competition. The dataset includes demographic, behavioural, health-related, and opinion-based features about individuals' attitudes and actions regarding flu vaccination. Variables include both numerical (e.g. levels of concern or knowledge) and categorical data (e.g. age group, education, race, employment status)."
        },
        "What is the source of the training data?": {
          "answer": "The training data originates from the DrivenData competition 'Flu Shot Learning: Predict H1N1 and Seasonal Flu Vaccines'. It is a publicly available dataset compiled from a U.S. national survey conducted by the National 2009 H1N1 Flu Survey (NHFS)."
        },
        "How were the labels/ground-truth produced?": {
          "answer": "The labels were derived from survey responses indicating whether each participant received the H1N1 and/or seasonal flu vaccine. These were binary variables: '1' for respondents who reported receiving the vaccine and '0' for those who did not. The ground-truth therefore reflects self-reported vaccination status."
        },
        "What is the sample size of the training data?": {
          "answer": "The training dataset consists of 26,707 observations (individual survey respondents) and 37 features. This includes both predictor variables and the two target labels for H1N1 and seasonal flu vaccination."
        },
        "What dataset(s) is the system NOT using?": {
          "answer": "The system does not use any external health or medical record datasets, nor does it include real-time epidemiological or clinical data. It also excludes the separate test dataset provided by DrivenData for competition scoring, which lacks labels."
        },
        "What are the potential limitations/biases of the data?": {
          "answer": "The dataset is based on self-reported survey responses, which may introduce recall bias or social desirability bias. Certain demographic groups may be underrepresented, such as individuals without internet access or those less likely to participate in surveys. Additionally, missing values in variables like health insurance and employment could affect model performance if not properly imputed. The data also reflects attitudes and behaviours specific to the 2009 H1N1 pandemic, which may limit its applicability to other contexts or time periods."
        }
    }
  }
}

In [ ]:
# Example: comparing one question
for question in xaiqb_human["questions"]["Data"].keys():

    chatgpt_ans = xaiqb_chatgpt["questions"]["Data"][question]["answer"]
    human_ans = xaiqb_human["questions"]["Data"][question]["answer"]

    # 1. BERTScore
    P, R, F1 = score([chatgpt_ans], [human_ans], lang="en")

    # # 2. Sentence-BERT cosine similarity
    # model = SentenceTransformer('all-MiniLM-L6-v2')
    # emb1, emb2 = model.encode(chatgpt_ans, convert_to_tensor=True), model.encode(human_ans, convert_to_tensor=True)
    # semantic_sim = util.cos_sim(emb1, emb2).item()

    # # 3. Readability
    # readability_chatgpt = flesch_reading_ease(chatgpt_ans)
    # readability_human = flesch_reading_ease(human_ans)

    print(f"BERTScore F1: {F1.mean():.3f}")
    # print(f"Semantic similarity: {semantic_sim:.3f}")
    # print(f"Readability (ChatGPT): {readability_chatgpt:.1f}, (Human): {readability_human:.1f}")


In [80]:
# pip install evaluate bert-score rouge-score sentence-transformers transformers
import re
import numpy as np
from evaluate import load
from bert_score import score as bertscore
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline

rouge = load("rouge")

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
nli = pipeline("text-classification", model="roberta-large-mnli", top_k=None)

def extract_numbers(text):
    # captures ints, decimals, and comma-separated thousands
    nums = re.findall(r"\b\d{1,3}(?:,\d{3})*(?:\.\d+)?\b", text)
    # normalize "26,707" -> 26707.0
    out = []
    for n in nums:
        out.append(float(n.replace(",", "")))
    return out

def rouge_l(pred, ref):
    return rouge.compute(predictions=[pred], references=[ref])["rougeL"]

def bert_f1(pred, ref):
    P, R, F1 = bertscore([pred], [ref], lang="en", rescale_with_baseline=True)
    return float(F1[0])

def embedding_cos(pred, ref):
    ep = embedder.encode(pred, convert_to_tensor=True, normalize_embeddings=True)
    er = embedder.encode(ref, convert_to_tensor=True, normalize_embeddings=True)
    return float(util.cos_sim(ep, er)[0][0])

def nli_label(premise, hypothesis):
    # premise entails hypothesis? (using MNLI framing)
    # We return the best label among entailment/neutral/contradiction
    out = nli({"text": premise, "text_pair": hypothesis})[0]
    # best = max(out, key=lambda x: x["score"])
    # return best["label"], float(best["score"])
    return out["label"], float(out["score"])

def numeric_match_rate(pred, ref):
    p = set(extract_numbers(pred))
    r = set(extract_numbers(ref))
    if len(r) == 0:
        return None
    return len(p & r) / len(r)

# Example scoring for one Q/A pair
def score_pair(pred, ref):
    entail_ph, conf_ph = nli_label(pred, ref)  # does pred entail ref?
    entail_hp, conf_hp = nli_label(ref, pred)  # does ref entail pred?

    return {
        "rougeL": rouge_l(pred, ref),
        "bertscore_f1": bert_f1(pred, ref),
        "embed_cos": embedding_cos(pred, ref),
        "nli_pred_entails_ref": entail_ph,
        "nli_ref_entails_pred": entail_hp,
        "numeric_match_rate": numeric_match_rate(pred, ref),
    }


Some weights of the model checkpoint at roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [ ]:
xaiqb_human = {
    "questions": {
        "Data": {
                "What kind of data was the system trained on?": 
                {
                    "question_interpretation": "What type(s) of data is being used by the system?",
                    "core_answer": "The data is tabular and provided in two csv files. One file contains the features and the other file contains the labels/ground-truth.",
                    "answer": "The data is tabular and provided in two csv files. One file contains the features including protected (e.g. sex, age, education, employment), behavioural (e.g. social distancing, mask wearing, hand washing), opinion (H1N1 concern, doctor vaccine reccomendation), health (e.g. chronic medial conditions, health insurance) and demographic (e.g. geographical region, employment industry) characteristics. The other file contains their labels/ground-truth which indicate if a respondant recieved the H1N1 vaccination or not (Yes = 1, No = 0)."
                },
                "What is the source of the training data?": 
                {
                    "question_interpretation": "Where does the training data come from?",
                    "core_answer": "The data is from DrivenData's 'Flu Shot Learning: Predict H1N1 and Seasonal Flu Vaccines' competition and originates from the US National 2009 H1N1 Flu Survey (NHFS).",
                    "answer": "Data is sourced from DrivenData's 'Flu Shot Learning: Predict H1N1 and Seasonal Flu Vaccines' competition and originates from the US National 2009 H1N1 Flu Survey (NHFS). The US National 2009 H1N1 Flu Survey (NHFS) was conducted between October 2009 and June 2010. It was a one-time telephone survey of households to monitor vaccinations during the 2009-2010 flu season in response to the 2009 H1N1 pandemic."
                },
                "How were the labels/ground-truth produced?": 
                {
                    "question_interpretation": "Where have the validated answers/responses come from? How were they created?",
                    "core_answer": "Respondants were asked during a telephone survey if they had received the H1N1 vaccination or not. This is a binary feature (Yes = 1, No = 0).",
                    "answer": "As part of the US National 2009 H1N1 Flu Survey (NHFS) a telephone sruvey was conducted. Respondants self reported if they had recieved the H1N1 vaccination. This is a binary feature indicating if they recieved the H1N1 vaccination or not (Yes = 1, No = 0)."
                },
                "What is the sample size of the training data?": 
                {
                    "question_interpretation": "How much data is being used to train the system?",
                    "core_answer": "The training data is 21,365 rows with 37 features.",
                    "answer": "There are a total of 26,707 observations with 37 features. This was split into training and testing data using an 80:20 split. Training data is 21,365 rows, testing data is 5,342 rows."
                },
                "What dataset(s) is the system NOT using?": 
                {
                    "question_interpretation": "What data is the system limited to using?",
                    "core_answer": "The system is only using data provided by the US National 2009 H1N1 Flu Survey (NHFS).",
                    "answer": "The system is not using any additional data such as public health datasets or electronic health records. Only the survey data provided by the US National 2009 H1N1 Flu Survey (NHFS) is being used by the system."
                },
                "What are the potential limitations/biases of the data?": 
                {
                    "question_interpretation": "What considerations need to be made when using this data which may (negatively) effect system performance?",
                    "core_answer": "The data is not representative of a large range of demographics. It is limited to persons 6 months or older living in the United States with telephone access.",
                    "answer": "The interviews were conducted over telephone and all respondants were persons 6 months or older living in the United States. These respondants were self reporting on their vaccination status of the H1N1 vaccine. This limits the representation of the overall (global) population and the attitudes towards the H1N1 vaccination. This means the data has limited transferability, and cannot be used to effectively model other countries and/or demographics."
                }
        }
    }
}

In [11]:
xaiqb_gemini = {
"questions": {
"Data": {
"What kind of data was the system trained on?": {
"question_interpretation": "This question asks about the type and nature of the data used to train the AI system. It seeks to understand the features and variables included in the dataset.",
"answer": "The system was trained on data from a telephone survey regarding vaccination uptake for H1N1 and seasonal flu. The data contains 38 features, including respondent demographics, behaviours, health status, and opinions on vaccination. The target variables are `h1n1_vaccine` and `seasonal_vaccine`, which indicate whether a respondent received the respective vaccine.【bac2aade-f9f5-4cc6-9af0-048acf0af6af-1】【bac2aade-f9f5-4cc6-9af0-048acf0af6af-11】"
},
"What is the source of the training data?": {
"question_interpretation": "This question seeks to identify the origin of the data used for training the AI model.",
"answer": "The data was sourced from a public competition on the DrivenData platform, titled 'Flu Shot Learning: Predict H1N1 and Seasonal Flu Vaccines'. The data was originally collected via a national telephone survey in the United States by the National Center for Immunization and Respiratory Diseases (NCIRD).【bac2aade-f9f5-4cc6-9af0-048acf0af6af-0】"
},
"How were the labels/ground-truth produced?": {
"question_interpretation": "This question asks how the correct answers (labels) for the training data were obtained.",
"answer": "The labels, indicating whether a respondent received the H1N1 or seasonal flu vaccine, were self-reported by the survey participants.【bac2aade-f9f5-4cc6-9af0-048acf0af6af-1】"
},
"What is the sample size of the training data?": {
"question_interpretation": "This question asks about the number of records or observations in the training dataset.",
"answer": "The training dataset consists of 26,707 survey responses.【bac2aade-f9f5-4cc6-9af0-048acf0af6af-11】"
},
"What dataset(s) is the system NOT using?": {
"question_interpretation": "This question aims to identify any data that was explicitly excluded from the model training process.",
"answer": "The modelling process for predicting H1N1 vaccine uptake intentionally excludes all features related to the seasonal flu vaccine.【bac2aade-f9f5-4cc6-9af0-048acf0af6af-36】"
},
"What are the potential limitations/biases of the data?": {
"question_interpretation": "This question asks about any potential shortcomings or biases present in the training data that could affect the model's performance or fairness.",
"answer": "The dataset has several potential limitations and biases. Firstly, there is a significant amount of missing data for several features, notably for `health_insurance` (46% missing), `employment_industry` (50% missing), and `employment_occupation` (50% missing). Secondly, the data is self-reported, which may introduce recall and social desirability biases. Thirdly, some categorical features, such as `hhs_geo_region`, `employment_industry`, and `employment_occupation`, are anonymised, which limits their interpretability. Finally, there is a class imbalance in the target variable, with only 21% of respondents having received the H1N1 vaccine, which could potentially bias the model towards predicting non-vaccination.【bac2aade-f9f5-4cc6-9af0-048acf0af6af-20】【bac2aade-f9f5-4cc6-9af0-048acf0af6af-23】【bac2aade-f9f5-4cc6-9af0-048acf0af6af-22】【bac2aade-f9f5-4cc6-9af0-048acf0af6af-18】"
}
}
}
}

In [ ]:
xaiqb_chatgpt = {
  "questions": {
    "Data": {
        "What kind of data was the system trained on?": {
          "question_interpretation": "This question seeks to understand the nature and structure of the data used to train the AI model, including the types of variables and the kind of information they represent.",
          "answer": "The system was trained on survey data from the DrivenData 'Flu Shot Learning: Predict H1N1 and Seasonal Flu Vaccines' competition. The dataset includes demographic, behavioural, health-related, and opinion-based features about individuals' attitudes and actions regarding flu vaccination. Variables include both numerical (e.g. levels of concern or knowledge) and categorical data (e.g. age group, education, race, employment status)."
        },
        "What is the source of the training data?": {
          "question_interpretation": "This question aims to identify where the data originated from and whether it is publicly available or proprietary.",
          "answer": "The training data originates from the DrivenData competition 'Flu Shot Learning: Predict H1N1 and Seasonal Flu Vaccines'. It is a publicly available dataset compiled from a U.S. national survey conducted by the National 2009 H1N1 Flu Survey (NHFS)."
        },
        "How were the labels/ground-truth produced?": {
          "question_interpretation": "This question asks how the target outcomes (labels) were defined and verified, which is crucial for assessing the reliability of the model's predictions.",
          "answer": "The labels were derived from survey responses indicating whether each participant received the H1N1 and/or seasonal flu vaccine. These were binary variables: '1' for respondents who reported receiving the vaccine and '0' for those who did not. The ground-truth therefore reflects self-reported vaccination status."
        },
        "What is the sample size of the training data?": {
          "question_interpretation": "This question seeks to determine how many data points (participants) were used to train the model, which affects the model's robustness and generalisability.",
          "answer": "The training dataset consists of 26,707 observations (individual survey respondents) and 37 features. This includes both predictor variables and the two target labels for H1N1 and seasonal flu vaccination."
        },
        "What dataset(s) is the system NOT using?": {
          "question_interpretation": "This question clarifies which datasets were excluded from training to avoid confusion about the model's data sources.",
          "answer": "The system does not use any external health or medical record datasets, nor does it include real-time epidemiological or clinical data. It also excludes the separate test dataset provided by DrivenData for competition scoring, which lacks labels."
        },
        "What are the potential limitations/biases of the data?": {
          "question_interpretation": "This question explores possible weaknesses or biases in the dataset that could affect the fairness or accuracy of the model's predictions.",
          "answer": "The dataset is based on self-reported survey responses, which may introduce recall bias or social desirability bias. Certain demographic groups may be underrepresented, such as individuals without internet access or those less likely to participate in surveys. Additionally, missing values in variables like health insurance and employment could affect model performance if not properly imputed. The data also reflects attitudes and behaviours specific to the 2009 H1N1 pandemic, which may limit its applicability to other contexts or time periods."
        }
    }
  }
}


In [9]:
xaiqb_claude = {
  "questions": {
    "Data": {
        "What kind of data was the system trained on?": {
          "question_interpretation": "This question asks about the nature and type of information used to train the AI model, including what variables or features were collected and what format the data takes.",
          "answer": "The system was trained on survey response data related to H1N1 and seasonal flu vaccine uptake. The dataset contains 37 features covering several categories: behavioural factors (such as whether respondents took antiviral medications, avoided close contact with symptomatic individuals, bought face masks, washed hands frequently, reduced time at large gatherings, and avoided touching their face); health-related factors (including doctor recommendations for vaccines, chronic medical conditions, contact with infants under 6 months, healthcare worker status, and health insurance coverage); opinion-based factors (respondents' views on vaccine effectiveness, perceived risk of illness, and concerns about getting sick from the vaccine); and demographic factors (age group, education level, ethnicity, sex, income relative to poverty thresholds, marital status, housing situation, employment status, geographic region, and household composition). The target variable is whether the respondent received the H1N1 flu vaccine (binary: yes or no)."
        },
        "What is the source of the training data?": {
          "question_interpretation": "This question seeks to understand where the data originated from, including the organisation or study that collected it and any relevant context about how it was gathered.",
          "answer": "The training data comes from DrivenData's competition titled 'Flu Shot Learning: Predict H1N1 and Seasonal Flu Vaccines'. The original data is derived from the National 2009 H1N1 Flu Survey (NHFS), which was conducted in the United States. The survey collected information about respondents' backgrounds, opinions, and health behaviours related to the H1N1 pandemic and seasonal influenza."
        },
        "How were the labels/ground-truth produced?": {
          "question_interpretation": "This question asks how the correct answers (labels) that the model learns from were determined, including whether they were self-reported, verified by experts, or derived through some other process.",
          "answer": "The labels were produced through self-reporting by survey respondents. Participants in the National 2009 H1N1 Flu Survey indicated whether they had received the H1N1 flu vaccine (coded as 0 for 'No' and 1 for 'Yes') and whether they had received the seasonal flu vaccine. These self-reported vaccination statuses serve as the ground-truth labels for training the predictive model."
        },
        "What is the sample size of the training data?": {
          "question_interpretation": "This question asks about the number of individual records or observations in the dataset used to train the model, which affects the model's ability to learn patterns reliably.",
          "answer": "The training dataset contains 26,707 observations (respondents) with 37 features. For model training, an 80:20 train-test split was applied, meaning approximately 21,366 observations were used for training and 5,341 for testing. The model was validated using 5-fold cross-validation during hyperparameter tuning."
        },
        "What dataset(s) is the system NOT using?": {
          "question_interpretation": "This question asks about data sources or types of information that were deliberately excluded or not available for training, which helps understand the boundaries of what the model has learned.",
          "answer": "The system does not use clinical or medical record data—it relies solely on self-reported survey responses rather than verified health records. It does not incorporate real-time or longitudinal data tracking changes in attitudes or behaviours over time. The model also excludes seasonal vaccine-related features when predicting H1N1 vaccine uptake specifically. Additionally, the system does not use data from outside the United States or from time periods other than the 2009 H1N1 pandemic context."
        },
        "What are the potential limitations/biases of the data?": {
          "question_interpretation": "This question asks about known or potential issues with the data that could affect the model's fairness, accuracy, or generalisability, including systematic errors or underrepresentation of certain groups.",
          "answer": "Several potential limitations and biases exist in the data. First, there is significant missing data: the health insurance variable has 45.96% missing values, employment industry has 49.91% missing, and employment occupation has 50.44% missing. Second, self-reporting bias may affect accuracy, as respondents might misremember or misrepresent their vaccination status, opinions, or behaviours. Third, there is demographic imbalance: the most common age group is 65+ years (representing 74.38% of the top category), and the majority of respondents are White (79.46%). Fourth, the data reflects a specific historical context (the 2009 H1N1 pandemic in the United States), which may limit generalisability to other populations, time periods, or health crises. Fifth, some geographic identifiers are anonymised (using codes like 'oxchjgsf' for regions), which limits the ability to assess geographic representativeness."
        }
    }
  }
}


##### Similarity

In [15]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
for category in xaiqb_human["questions"].keys():  
        for question in xaiqb_human["questions"][category].keys():
            
            question_interpretations = [
                xaiqb_human["questions"][category][question]["question_interpretation"],
                xaiqb_claude["questions"][category][question]["question_interpretation"],
                xaiqb_chatgpt["questions"][category][question]["question_interpretation"],
                xaiqb_gemini["questions"][category][question]["question_interpretation"]
                ]
            
            question_interpretation_encoding = model.encode(question_interpretations)

            human_question_interpretation_cosine_similarity = util.cos_sim(question_interpretation_encoding[0], question_interpretation_encoding[1:]).tolist()[0]

In [25]:
import re
re.findall("\w+", xaiqb_human["questions"][category][question]["question_interpretation"])

['What',
 'considerations',
 'need',
 'to',
 'be',
 'made',
 'when',
 'using',
 'this',
 'data',
 'which',
 'may',
 'negatively',
 'effect',
 'system',
 'performance']

In [ ]:
import nltk

hypothesis = ['It', 'is', 'a', 'cat', 'at', 'room']
reference = ['It', 'is', 'a', 'cat', 'inside', 'the', 'room']
#there may be several references
BLEUscore = nltk.translate..sentence_bleu([reference], hypothesis)
print(BLEUscore)

0.4548019047027907


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "gpt2"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretraineWhad(model_name)

Perplexity: 10.321878433227539


In [ ]:

inputs = tokenizer("Hello, how are you?", return_tensors="pt")
inputs["labels"] = inputs["input_ids"].clone()

with torch.no_grad():
    outputs = model(**inputs)
    loss = outputs.loss  # This is the Cross-Entropy Loss
    perplexity = torch.exp(loss)

print(f"Perplexity: {perplexity.item()}")

In [ ]:
nltk.translate.bleu_score.sentence_bleu(
    [re.findall("\w+", "Hello, how are you doing today?")],
    re.findall("\w+", "Hello, how are you doing today? Hello, how are you doing today? Hello, how are you doing today?"),
    
)

0.26460159523593296

In [27]:
[nltk.translate.bleu_score.sentence_bleu([re.findall("\w+", xaiqb_human["questions"][category][question]["question_interpretation"])], re.findall("\w+", xaiqb_claude["questions"][category][question]["question_interpretation"])) 
 for question in xaiqb_human["questions"][category].keys() for category in xaiqb_human["questions"].keys()]

/opt/miniconda3/envs/venv_thesis/lib/python3.11/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/opt/miniconda3/envs/venv_thesis/lib/python3.11/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/opt/miniconda3/envs/venv_thesis/lib/python3.11/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider usi

[1.1739546629120287e-231,
 1.0518351895246305e-231,
 1.1640469867513693e-231,
 0.08055259301867158,
 9.257324954728539e-232,
 7.850707993042515e-232]

In [ ]:
import nltk

hypothesis = xaiqb_human["questions"][category][question]["question_interpretation"]
reference = ['It', 'is', 'a', 'cat', 'inside', 'the', 'room']
#there may be several references
BLEUscore = nltk.translate.bleu_score.sentence_bleu([reference], hypothesis)
print(BLEUscore)

In [ ]:
def pull_similariy_scores(model):
    similarity_scores = []
    for category in xaiqb_human["questions"].keys():  
        for question in xaiqb_human["questions"][category].keys():

            answers = [
                xaiqb_human["questions"][category][question]["answer"],
                xaiqb_claude["questions"][category][question]["answer"],
                xaiqb_chatgpt["questions"][category][question]["answer"],
                xaiqb_gemini["questions"][category][question]["answer"]
                ]
            
            question_interpretations = [
                xaiqb_human["questions"][category][question]["question_interpretation"],
                xaiqb_claude["questions"][category][question]["question_interpretation"],
                xaiqb_chatgpt["questions"][category][question]["question_interpretation"],
                xaiqb_gemini["questions"][category][question]["question_interpretation"]
                ]
            
            answer_encoding = model.encode(answers)

            human_answer_cosine_similarity      = util.cos_sim(answer_encoding[0], answer_encoding[1:]).tolist()[0]
            claude_answer_cosine_similarity     = util.cos_sim(answer_encoding[1], answer_encoding[2:]).tolist()[0]
            chatgpt_answer_cosine_similarity    = util.cos_sim(answer_encoding[2], answer_encoding[3:]).tolist()[0]

            question_interpretation_encoding = model.encode(question_interpretations)

            human_question_interpretation_cosine_similarity     = util.cos_sim(question_interpretation_encoding[0], question_interpretation_encoding[1:]).tolist()[0]
            claude_question_interpretation_cosine_similarity    = util.cos_sim(question_interpretation_encoding[1], question_interpretation_encoding[2:]).tolist()[0]
            chatgpt_question_interpretation_cosine_similarity   = util.cos_sim(question_interpretation_encoding[2], question_interpretation_encoding[3:]).tolist()[0]

            similarity_scores += [pd.DataFrame({
                "Category": category,
                "Question": question,

                "Answer Human Claude"    : [human_answer_cosine_similarity[0]],
                "Answer Human ChatGPT"   : [human_answer_cosine_similarity[1]],
                "Answer Human Gemini"    : [human_answer_cosine_similarity[2]],
                "Answer Claude ChatGPT"  : [claude_answer_cosine_similarity[0]],
                "Answer Claude Gemini"   : [claude_answer_cosine_similarity[1]],
                "Answer ChatGPT Gemini"  : [chatgpt_answer_cosine_similarity[0]],

                "Question Human Claude"    : [human_question_interpretation_cosine_similarity[0]],
                "Question Human ChatGPT"   : [human_question_interpretation_cosine_similarity[1]],
                "Question Human Gemini"    : [human_question_interpretation_cosine_similarity[2]],
                "Question Claude ChatGPT"  : [claude_question_interpretation_cosine_similarity[0]],
                "Question Claude Gemini"   : [claude_question_interpretation_cosine_similarity[1]],
                "Question ChatGPT Gemini"  : [chatgpt_question_interpretation_cosine_similarity[0]],
w
                })
                ]
    
    similarity_scores_df = pd.concat(similarity_scores)
    # columns = [("", "Category"),("", "Question")] + [("Answer Similarity", x) for x in ["Human Claude", "Human ChatGPT", "Human Gemini", "Claude ChatGPT", "Claude Gemini", "ChatGPT Gemini"]] + [("Question Interpretation Similarity", x) for x in ["Human Claude", "Human ChatGPT"]]#, "Human Gemini", "Claude ChatGPT", "Claude Gemini", "ChatGPT Gemini"]]
    # similariy_scores_df.columns = pd.MultiIndex.from_tuples(columns)
    return similarity_scores_df

In [12]:
# all-mpnet-base-v2, all-MiniLM-L12-v2, all-MiniLM-L6-v2
similarity_scores = pull_similariy_scores(SentenceTransformer("all-MiniLM-L6-v2"))
similarity_scores.head()

,Category,Question,Answer Human Claude,Answer Human ChatGPT,Answer Human Gemini,Answer Claude ChatGPT,Answer Claude Gemini,Answer ChatGPT Gemini,Question Human Claude,Question Human ChatGPT,Question Human Gemini,Question Claude ChatGPT,Question Claude Gemini,Question ChatGPT Gemini
0,Data,What kind of data was the system trained on?,0.700254,0.586283,0.713351,0.829127,0.824664,0.800059,0.448703,0.398097,0.442935,0.888310,0.899813,0.873998
0,Data,What is the source of the training data?,0.925928,0.898430,0.865445,0.941646,0.917245,0.917246,0.554711,0.508531,0.721711,0.661915,0.527722,0.500023
0,Data,How were the labels/ground-truth produced?,0.738515,0.794223,0.794289,0.895055,0.874978,0.929273,0.515370,0.287950,0.403139,0.628577,0.727458,0.470209
0,Data,What is the sample size of the training data?,0.815627,0.593949,0.541490,0.581263,0.602275,0.621485,0.538312,0.434629,0.542028,0.588072,0.779158,0.489643
0,Data,What dataset(s) is the system NOT using?,0.699161,0.615587,0.504966,0.636618,0.682334,0.286218,0.382508,0.317807,0.333089,0.788353,0.792222,0.813905


In [13]:
similarity_scores.to_clipboard(index = False)

In [284]:
import itertools, re
answer_metrics = [f"Answer {x} {y}" for x, y in itertools.combinations(["Human", "Claude", "ChatGPT", "Gemini"], 2)]
question_metrics = [f"Question {x} {y}" for x, y in itertools.combinations(["Human", "Claude", "ChatGPT", "Gemini"], 2)]


In [283]:
to_plot = similarity_scores.groupby("Category").mean(numeric_only = True).reset_index().melt(id_vars = "Category")

In [285]:
to_plot[to_plot["variable"].apply(lambda x: re.search("Answer Human", x) is not None)]

,Category,variable,value
0,Data,Answer Human Claude,0.743570
1,How (global model-wide explanation),Answer Human Claude,0.618163
2,Others,Answer Human Claude,0.522952
3,Output,Answer Human Claude,0.569730
4,Performance,Answer Human Claude,0.598300
5,UI,Answer Human Claude,0.583885
6,Data,Answer Human ChatGPT,0.641748
7,How (global model-wide explanation),Answer Human ChatGPT,0.541211
8,Others,Answer Human ChatGPT,0.384738
9,Output,Answer Human ChatGPT,0.511804


In [279]:
px.line_polar(to_plot[to_plot["variable"].apply(lambda x: re.search("Answer Human", x) is not None)], 
              r = "value", range_r = [0, 1], theta = "Category", color = "variable", line_close = True, 
              height = 500, width = 1000,
              title = "Semantic Similarity Scores of Human vs LLM Generated Answers").update_layout(legend = dict(orientation = "h"), title_x = 0.5)

In [280]:
px.line_polar(to_plot[to_plot["variable"].apply(lambda x: re.search("Question Human", x) is not None)], 
              r = "value", range_r = [0, 1], theta = "Category", color = "variable", line_close = True, 
              height = 500, width = 1000,
              title = "Semantic Similarity Scores of Human vs LLM Interpretations of Questions").update_layout(legend = dict(orientation = "h"), title_x = 0.5)

In [ ]:
for i in range(len(answer_metrics)):
    print(f"{answer_metrics[i]}: {similarity_scores[[answer_metrics[1], question_metrics[i]]].corr().iloc[0, 1]:.3f}")

Answer Human Claude: 0.008
Answer Human ChatGPT: 0.202
Answer Human Gemini: -0.060
Answer Claude ChatGPT: -0.102
Answer Claude Gemini: 0.326
Answer ChatGPT Gemini: 0.264


In [ ]:
largest_similarity = similarity_scores.nlargest(3, "similarity").reset_index(drop = True)
largest_similarity

,category,question,similarity
0,Data,How were the labels/ground-truth produced?,0.757691
1,Performance,What kind of mistakes is the system likely to ...,0.720586
2,How (global model-wide explanation),What are the top rules/features that determine...,0.719658


In [ ]:
largest_similarity = similarity_scores.nlargest(3, "similarity").reset_index(drop = True)
largest_similarity

,category,question,similarity
0,How (global model-wide explanation),What are the top rules/features that determine...,0.825490
1,Data,What is the source of the training data?,0.813104
2,Performance,What kind of mistakes is the system likely to ...,0.799797


In [ ]:
for category, question, similarity in largest_similarity.values:
    print(f'''
    {category} | {question}
    Similarity score: {similarity:.3f}
    {"-" * 80}
    Human:\t{xaiqb_human[category]["question_answer"][question]["answer"]}
    LLM:\t{xaiqb_gemini[category]["question_answer"][question]["answer"]}
    ''')


    Data | How were the labels/ground-truth produced?
    Similarity score: 0.758
    --------------------------------------------------------------------------------
    Human:	Respondants were asked if they had recieved the H1N1 vaccination or not. This is a binary 0/1 feature indicating no/yes respectively.
    LLM:	The 'ground truth' for our model is whether a person received the H1N1 vaccine. This information was self-reported by the survey respondents.
    

    Performance | What kind of mistakes is the system likely to make?
    Similarity score: 0.721
    --------------------------------------------------------------------------------
    Human:	The system will incorrectly identify if a person gets the vaccine or not (i.e. False Positive or False Negative).
    LLM:	The most common mistake is a 'false positive', where the model predicts an individual will get vaccinated, but they do not. It is less likely to make a 'false negative' error, where it predicts someone will not ge

In [ ]:
for category, question, similarity in largest_similarity.values:
    print(f'''
    {category} | {question}
    Similarity score: {similarity:.3f}
    {"-" * 80}
    Human:\t{xaiqb_human["Global"][category]["question_answer"][question]}
    LLM:\t{xaiqb_gemini["Global"][category]["question_answer"][question]}
    ''')


    How (global model-wide explanation) | What are the top rules/features that determine its predictions?
    Similarity score: 0.825
    --------------------------------------------------------------------------------
    Human:	Top features are if the doctor reccomends the H1N1 vaccine, if they believe the vaccine is effective, and how much of a risk they believe H1N1 is.
    LLM:	The top three most influential features are whether a doctor recommended the H1N1 vaccine, the respondent's perceived risk of getting sick from H1N1, and their opinion on the vaccine's effectiveness.【2-34】
    

    Data | What is the source of the training data?
    Similarity score: 0.813
    --------------------------------------------------------------------------------
    Human:	Data comes from the National 2009 H1N1 Flu Survey (NHFS).
    LLM:	The data comes from the 'Flu Shot Learning: Predict H1N1 and Seasonal Flu Vaccines' competition on DrivenData, which is based on the National 2009 H1N1 Flu Su

In [ ]:
smallest_similarity = similarity_scores.nsmallest(3, "similarity").reset_index(drop = True)
smallest_similarity

,category,question,similarity
0,How (global model-wide explanation),How does it weigh different features?,0.072991
1,Output,What is the scope of the output data?,0.188284
2,Performance,What are the limitations of the system?,0.218875


In [ ]:
smallest_similarity = similarity_scores.nsmallest(3, "similarity").reset_index(drop = True)
smallest_similarity

,category,question,similarity
0,Data,What dataset(s) is the system NOT using?,0.097394
1,Data,What are the potential limitations/biases of t...,0.118485
2,Output,What is the scope of the output data?,0.129578


In [ ]:
for category, question, similarity in smallest_similarity.values:
    print(f'''
    {category} | {question}
    Similarity score: {similarity:.3f}
    {"-" * 80}
    Human:\t{xaiqb_human[category]["question_answer"][question]["answer"]}
    LLM:\t{xaiqb_gemini[category]["question_answer"][question]["answer"]}
    ''')


    How (global model-wide explanation) | How does it weigh different features?
    Similarity score: 0.073
    --------------------------------------------------------------------------------
    Human:	Feature weights are not assigned when using a RandomForestClassifier(). Instead, Gini (impurity) is used to measure the quality of a split in a decision tree, which is how 'mixed' the classes are in a node. Lower impurity means the node is more pure (mostly one class). For a binary classifier, this impurity is measured from [0, 0.5].
    LLM:	The most influential factor is whether a doctor recommended the H1N1 vaccine. Opinions about the vaccine's effectiveness and the risk of H1N1 infection are also highly weighted. Demographic factors like age and race have a smaller, but still significant, influence.
    

    Output | What is the scope of the output data?
    Similarity score: 0.188
    --------------------------------------------------------------------------------
    Human:	The

In [ ]:
for category, question, similarity in smallest_similarity.values:
    print(f'''
    {category} | {question}
    Similarity score: {similarity:.3f}
    {"-" * 80}
    Human:\t{xaiqb_human["Global"][category]["question_answer"][question]}
    LLM:\t{xaiqb_gemini["Global"][category]["question_answer"][question]}
    ''')


    Data | What dataset(s) is the system NOT using?
    Similarity score: 0.097
    --------------------------------------------------------------------------------
    Human:	Only data provided by these two csv files are being used by the system.
    LLM:	The model is trained specifically on H1N1-related features and excludes features related to the seasonal flu vaccine to focus the prediction.【2-66】
    

    Data | What are the potential limitations/biases of the data?
    Similarity score: 0.118
    --------------------------------------------------------------------------------
    Human:	Limited transferability, as all respondants were persons 6 months or older living in the United States at the time of the interview, over telephone.
    LLM:	The dataset has a significant number of missing values in key fields like health insurance, employment industry, and occupation, which could introduce bias. For example, nearly 46% of respondents have missing health insurance information.【2

#### Readability Metric Descriptions

##### Flesch-Kincaid Grade Level
The U.S. Army uses Flesch-Kincaid Grade Level for assessing the difficulty of technical manuals. The commonwealth of Pennsylvania uses Flesch-Kincaid Grade Level for scoring automobile insurance policies to ensure their texts are no higher than a ninth grade level of reading difficulty. Many other U.S. states also use Flesch-Kincaid Grade Level to score other legal documents such as business policies and financial forms.

##### Flesch Reading Ease
The U.S. Department of Defense uses the Reading Ease test as the standard test of readability for its documents and forms. Florida requires that life insurance policies have a Flesch Reading Ease score of 45 or greater.

##### Dale Chall Readability
The Dale-Chall Formula is an accurate readability formula for the simple reason that it is based on the use of familiar words, rather than syllable or letter counts. Reading tests show that readers usually find it easier to read, process and recall a passage if they find the words familiar.

##### Automated Readability Index (ARI)
Unlike the other indices, the ARI, along with the Coleman-Liau, relies on a factor of characters per word, instead of the usual syllables per word. ARI is widely used on all types of texts.

##### Coleman Liau Index
The Coleman-Liau Formula usually gives a lower grade value than any of the Kincaid, ARI and Flesch values when applied to technical documents.

##### Gunning Fog
The Gunning fog index measures the readability of English writing. The index estimates the years of formal education needed to understand the text on a first reading. A fog index of 12 requires the reading level of a U.S. high school senior (around 18 years old).

##### SMOG
The SMOG Readability Formula (Simple Measure of Gobbledygook) is a popular method to use on health literacy materials.

##### SPACHE
The Spache Readability Formula is used for Primary-Grade Reading Materials, published in 1953 in The Elementary School Journal. The Spache Formula is best used to calculate the difficulty of text that falls at the 3rd grade level or below.

##### Linsear Write
Linsear Write is a readability metric for English text, purportedly developed for the United States Air Force to help them calculate the readability of their technical manuals.